In [1]:
# cell 1
# Install runtime dependencies.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy accelerate safetensors

# Remove optional packages that may break transformers/vLLM imports.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Transformers is useful for tokenizer checks and compatibility.
!uv pip install --system -U "transformers>=4.51.0"

# Recent vLLM nightly for CUDA 13 / Blackwell.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 68 packages in 119ms
Prepared 8 packages in 0.33ms
Uninstalled 8 packages in 129ms
Installed 8 packages in 115ms
 - numpy==2.3.5
 + numpy==2.4.6
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.0
 - triton==3.6.0
 + triton==3.7.0
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 48ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 99ms
Checked 27 packages in 0.28ms
Using Python 3.12.13 environment at: /usr
Resolved 190 packages in 2.66s
Prepared 11 packages in 6.57s
Uninstalled 9 packages in 137ms
Installed 11 packages in 112ms
 - numpy==2.4.6
 + numpy==2.3.5
 - nvidia-cublas

In [2]:
#cell 2
# Imports and global config.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback

from pathlib import Path
from tqdm.auto import tqdm
from openai import OpenAI
from jsonschema import validate

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DATASET = "2wikimultihopqa"

LLM_MODEL_NAME = "Qwen/Qwen3.5-27B"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Match the traversal notebook setting and avoid very large thinking-context overhead.
MAX_MODEL_LEN = 32768

# Match the traversal notebook setting.
GPU_MEMORY_UTILIZATION = 0.92

MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = 32768

SERVER_LOG_PATH = Path("/content/vllm_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_answer_server.pid")

LOCAL_RUNTIME_DIR = Path("/content/final_project_copy")
LOCAL_EVIDENCE_DIR = LOCAL_RUNTIME_DIR / "evidence" / DATASET
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
DRIVE_IDEA_DIR = DRIVE_PROJECT_DIR / "idea_1"

DRIVE_EVIDENCE_PATH = (
    DRIVE_IDEA_DIR
    / "evidence"
    / DATASET
    / "2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json"
)

LOCAL_EVIDENCE_PATH = LOCAL_EVIDENCE_DIR / DRIVE_EVIDENCE_PATH.name

DRIVE_ANSWER_DIR = DRIVE_IDEA_DIR / "answers" / DATASET

DRIVE_ANSWER_PATH = DRIVE_ANSWER_DIR / "2wikimultihopqa_answer_qwen3.5.json"

ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None  # None means all records.

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25

# No-thinking mode needs only a short JSON answer.
ANSWER_MAX_TOKENS = 1024

# Retry only needs enough space to repair JSON, not a large thinking budget.
ANSWER_RETRY_MAX_TOKENS = 2048

# Used only for logging retry diagnostics after invalid JSON.
LOW_COMPLETION_TOKENS_THRESHOLD = 128

print("Model:", LLM_MODEL_NAME)
print("Max model len:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Drive evidence path:", DRIVE_EVIDENCE_PATH)
print("Local evidence path:", LOCAL_EVIDENCE_PATH)
print("Answer output path:", DRIVE_ANSWER_PATH)

Model: Qwen/Qwen3.5-27B
Max model len: 32768
GPU memory utilization: 0.92
Drive evidence path: /content/drive/MyDrive/final_project/idea_1/evidence/2wikimultihopqa/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
Local evidence path: /content/final_project_copy/evidence/2wikimultihopqa/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
Answer output path: /content/drive/MyDrive/final_project/idea_1/answers/2wikimultihopqa/2wikimultihopqa_answer_qwen3.5.json


In [3]:
# cell 3
# Mount Google Drive and copy evidence to local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

DRIVE_ANSWER_DIR.mkdir(parents=True, exist_ok=True)

if not DRIVE_EVIDENCE_PATH.exists():
    raise FileNotFoundError(f"Evidence file not found: {DRIVE_EVIDENCE_PATH}")

def file_is_same_size(src: Path, dst: Path) -> bool:
    # Check whether local copy is complete.
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    # Copy with temp file to avoid partial local copies.
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print("Local evidence copy already exists.")
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

copy_file_to_local(DRIVE_EVIDENCE_PATH, LOCAL_EVIDENCE_PATH)

print("Evidence copied to local disk.")
print("Local evidence size MB:", LOCAL_EVIDENCE_PATH.stat().st_size / (1024 ** 2))
print("Answer directory:", DRIVE_ANSWER_DIR)

Mounted at /content/drive
Evidence copied to local disk.
Local evidence size MB: 10.814663887023926
Answer directory: /content/drive/MyDrive/final_project/idea_1/answers/2wikimultihopqa


In [4]:
# cell 4
# Load traversal evidence.

with open(LOCAL_EVIDENCE_PATH, "r", encoding="utf-8") as f:
    evidence_records = json.load(f)

if not isinstance(evidence_records, list):
    raise RuntimeError("Evidence JSON must be a list of records.")

required_keys = ["type", "question", "answer", "evidence_chunk", "evidence_path"]

for i, rec in enumerate(evidence_records[:5]):
    missing = [k for k in required_keys if k not in rec]
    if missing:
        print(f"Warning: record {i} missing keys:", missing)

print("Number of evidence records:", len(evidence_records))
print("First source_index:", evidence_records[0].get("source_index", 0))
print("First question:", evidence_records[0].get("question"))
print("First GT answer:", evidence_records[0].get("answer"))
print("First chunks:", len(evidence_records[0].get("evidence_chunk", [])))
print("First paths:", len(evidence_records[0].get("evidence_path", [])))

Number of evidence records: 1000
First source_index: 0
First question: Are North Marion High School (Oregon) and Seoul High School both located in the same country?
First GT answer: no
First chunks: 6
First paths: 5


In [5]:
# cell 5
# Start Qwen3.5 vLLM server in no-thinking mode.

def kill_process_tree(pid):
    # Kill a process and all children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    "--language-model-only",

    # No-thinking mode for Qwen.
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--enable-prefix-caching",
    "--generation-config", "vllm",
    "--dtype", "bfloat16",
    "--trust-remote-code",
]

server_env = os.environ.copy()

# Blackwell / CUDA 13 fixes.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 32768 --gpu-memory-utilization 0.92 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 1 --max-num-batched-tokens 32768 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 5389
Log: /content/vllm_answer_server.log


In [6]:
# cell 6
# Wait for vLLM server and create OpenAI-compatible client.

import requests

def tail_log(path, n=80):
    # Read last log lines.
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

LLM_SAMPLING_KWARGS = {
    "temperature": 0.6,
    "top_p": 0.95,
    "presence_penalty": 0.0,
}

LLM_EXTRA_BODY = {
    "top_k": 20,
    "min_p": 0.0,
    "repetition_penalty": 1.0,
    "chat_template_kwargs": {
        "enable_thinking": False,
    },
}

print("OpenAI-compatible client is ready.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=5879) INFO 06-12 12:25:12 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=5879) INFO 06-12 12:25:12 [parallel_state.py:1568] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:55827 backend=nccl
(EngineCore pid=5879) INFO 06-12 12:25:12 [parallel_state.py:1903] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=5879) INFO 06-12 12:25:12 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=5879) INFO 06-12 12:25:12 [gpu_model_runner.py:5087] Starting to load model Qwen/Qwen3.5-27B...
(EngineCore pid=5879) INFO 06-12 12:25:13 [cuda.py:433] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=

In [7]:
# cell 7
# Answer schema and prompt.

ANSWER_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "response": {
            "type": "string",
            "minLength": 1
        }
    },
    "required": ["response"]
}

ANSWER_SYSTEM_PROMPT = """
You are a careful and highly capable multi-hop open-domain question answering agent.

You will receive:
1. A 2WikiMultiHopQA-style question.
2. Final evidence chunks retrieved by a knowledge-graph traversal agent.
3. Final evidence paths retrieved from the same knowledge graph.

Knowledge graph structure:
- Node types:
  1. entity nodes
     - Entity nodes represent normalized entities.
  2. chunk nodes
     - Chunk nodes represent Wikipedia text chunks.

- Edge types:
  1. entity --relation--> entity
     - Directed edge from one entity to another entity.
     - The relation text is a short natural-language sentence describing the connection.
     - Each relation edge has a relation_id and a source chunk_id.
  2. entity --fact--> chunk
     - Directed edge from an entity to a Wikipedia chunk.
     - The fact text describes what information the chunk contains about that entity.
     - Each fact edge has a fact_id and a chunk_id.

Traversal summary:
- A traversal agent explored the knowledge graph to collect evidence for the question.
- The selected chunks may contain bridge evidence, comparison evidence, answer-bearing evidence, or irrelevant distractors.
- The selected paths may help connect entities across multiple hops.
- Some chunks and paths can be noisy or misleading. Treat them as possible distractors, not guaranteed evidence.
- Prefer explicit evidence from chunks, but use paths to guide multi-hop reasoning when they help connect facts.

Your task:
Answer the question using only the provided evidence chunks and evidence paths.

Reasoning instructions:
- Carefully analyze the question, evidence chunks, and evidence paths internally.
- Identify the relevant entities and the required reasoning type: bridge, comparison, or another multi-hop pattern.
- Connect facts across multiple chunks when needed.
- Use evidence paths as supporting signals for entity connections, but do not treat a path as stronger than explicit chunk text.
- Ignore distractor chunks or paths that are not needed for the answer.
- Do not use external knowledge.
- Do not guess beyond the provided evidence.

Answering rules:
1. Your primary goal is to synthesize the answer from the provided evidence.
2. Do not give up easily. Many questions require combining evidence from two or more chunks.
3. Only if the provided evidence is truly insufficient, contradictory, or does not support any answer after careful multi-hop reasoning, the final response must be exactly:
   Information not available
4. The final answer should be extremely concise.
5. Prefer a short answer phrase of 1 to 10 words when possible.
6. Never exceed 18 words unless the exact required response is Information not available.
7. Do not include explanations, citations, reasoning traces, or extra text in the final answer string.

Output format:
- The final assistant content must contain exactly one valid JSON object.
- The JSON object must match this schema:
{
  "response": "..."
}
- Do not wrap the JSON in markdown.
- Do not add any text before or after the JSON.
""".strip()

In [8]:
# cell 8
# Prompt formatting helpers.

def clean_text_for_prompt(text):
    # Preserve text content, only normalize Python None.
    if text is None:
        return ""
    return str(text)

def format_evidence_chunks(chunks):
    # Format all chunks without truncation.
    if not chunks:
        return "No evidence chunks provided."

    blocks = []

    for i, chunk in enumerate(chunks, start=1):
        title = clean_text_for_prompt(chunk.get("title", ""))
        text = clean_text_for_prompt(chunk.get("text", ""))

        block = (
            f"[Chunk {i}]\n"
            f"Title: {title}\n"
            f"Text:\n{text}"
        )
        blocks.append(block)

    return "\n\n".join(blocks)

def format_evidence_paths(paths):
    # Format all paths as p1, p2, ...
    if not paths:
        return "No evidence paths provided."

    lines = []

    for i, path_item in enumerate(paths, start=1):
        path_text = clean_text_for_prompt(path_item.get("path", ""))
        lines.append(f"p{i}: {path_text}")

    return "\n".join(lines)

def build_answer_prompt(record):
    # Build final QA prompt.
    question = clean_text_for_prompt(record.get("question", ""))

    chunks_text = format_evidence_chunks(record.get("evidence_chunk", []))
    paths_text = format_evidence_paths(record.get("evidence_path", []))

    prompt = f"""
Question:
{question}

Evidence chunks:
{chunks_text}

Evidence paths:
{paths_text}

Now answer the question. Return the final answer as one valid JSON object only.
""".strip()

    return prompt

# Preview one prompt.
preview_prompt = build_answer_prompt(evidence_records[0])
print(preview_prompt[:4000])
print("\nPrompt preview length in characters:", len(preview_prompt))

Question:
Are North Marion High School (Oregon) and Seoul High School both located in the same country?

Evidence chunks:
[Chunk 1]
Title: North Marion High School (Oregon)
Text:
North Marion High School is a public high school in Aurora, Oregon, United States. The school is part of the North Marion School District with all four schools being located on the same campus. The school draws students from the cities of Aurora, Hubbard, and Donald as well as the communities of Broadacres and Butteville.
Academics: The school earned a 4 (out of 5) in its report card grading for the 2013-2014 school year, meaning more than 70 percent of students met or exceeded standards on the Oregon Assessment of Knowledge and Skills. North Marion High School's completion rate was 94.8 percent, and its four-year graduation rate was 78.6 percent.
OAKS testing scores: 2013-2014 North Marion High School test scores: Reading: 95.7% passed or exceeded State Average: 85.6% Writing: 74.8% passed or exceeded State A

In [9]:
# cell 9
# JSON parsing and answer cleanup.

def strip_code_fence(text):
    # Remove markdown code fences.
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def strip_think_blocks(text):
    # Remove raw thinking blocks if returned.
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()

def extract_json_object(text):
    # Extract first JSON object from model output.
    text = strip_code_fence(strip_think_blocks(text))
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found. Raw text:\n{text[:2000]}")

    return text[start:end + 1]

def normalize_answer_text(answer):
    # Keep answer short and clean.
    answer = strip_think_blocks(answer)
    answer = strip_code_fence(answer)
    answer = re.sub(r"\s+", " ", answer).strip()

    if not answer:
        return "Information not available"

    # Remove accidental surrounding quotes.
    if len(answer) >= 2 and answer[0] == answer[-1] and answer[0] in ['"', "'"]:
        answer = answer[1:-1].strip()

    # Enforce exact fallback text.
    if answer.lower() in {
        "not available",
        "information unavailable",
        "unknown",
        "cannot determine",
        "not enough information",
        "insufficient information",
    }:
        return "Information not available"

    return answer

def parse_answer_json(content):
    # Parse answer JSON and validate schema.
    json_text = extract_json_object(content)
    parsed = json.loads(json_text)
    validate(instance=parsed, schema=ANSWER_SCHEMA)

    response = normalize_answer_text(parsed["response"])
    parsed["response"] = response

    return parsed

In [10]:
# cell 10
# LLM answer function with retry only for invalid or broken JSON.

def get_completion_tokens_from_usage(usage):
    # Read completion token count if vLLM/OpenAI-compatible usage provides it.
    if usage is None:
        return None

    try:
        usage_dict = usage.model_dump()
    except Exception:
        try:
            usage_dict = dict(usage)
        except Exception:
            return None

    return usage_dict.get("completion_tokens")


def should_retry_after_parse_error(response, content):
    # Retry only when the final output is not valid JSON.
    usage = response.usage
    completion_tokens = get_completion_tokens_from_usage(usage)

    finish_reason = None
    try:
        finish_reason = response.choices[0].finish_reason
    except Exception:
        pass

    low_completion = (
        completion_tokens is not None
        and completion_tokens < LOW_COMPLETION_TOKENS_THRESHOLD
    )

    hit_length_limit = finish_reason == "length"

    return {
        "retry": True,
        "completion_tokens": completion_tokens,
        "low_completion": low_completion,
        "finish_reason": finish_reason,
        "hit_length_limit": hit_length_limit,
    }


def call_llm_answer(prompt, max_retries=2):
    # Call no-thinking Qwen and parse short JSON answer.
    # Retry only if JSON parsing/validation fails.
    # Do not retry valid Information not available responses.
    last_error = None
    last_content = None
    last_retry_info = None

    messages = [
        {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    for attempt in range(max_retries + 1):
        current_messages = messages

        # First attempt uses a small answer budget.
        # Retries use a slightly larger budget only to repair JSON.
        current_max_tokens = ANSWER_MAX_TOKENS if attempt == 0 else ANSWER_RETRY_MAX_TOKENS

        if attempt > 0:
            repair_prompt = (
                prompt
                + "\n\nYour previous response could not be parsed as valid JSON. "
                + "Return exactly one valid JSON object like {\"response\": \"...\"}. "
                + "Do not wrap the JSON in markdown. "
                + "Do not add explanations, citations, reasoning traces, or extra text outside the JSON.\n\n"
                + f"Validation error:\n{last_error}\n\n"
                + f"Previous retry info:\n{json.dumps(last_retry_info, ensure_ascii=False)}\n\n"
                + f"Previous response:\n{last_content}"
            )

            current_messages = [
                {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                {"role": "user", "content": repair_prompt},
            ]

        response = client.chat.completions.create(
            model=LLM_MODEL_NAME,
            messages=current_messages,
            max_tokens=current_max_tokens,
            **LLM_SAMPLING_KWARGS,
            extra_body=LLM_EXTRA_BODY,
        )

        msg = response.choices[0].message
        content = msg.content or ""

        last_content = content

        usage_dict = response.usage.model_dump() if response.usage else None
        completion_tokens = get_completion_tokens_from_usage(response.usage)

        try:
            parsed = parse_answer_json(content)

            # Important:
            # If JSON is valid, return immediately.
            # This includes valid {"response": "Information not available"}.
            return {
                "response": parsed["response"],
                "raw_content": content,
                "usage": usage_dict,
                "completion_tokens": completion_tokens,
                "max_tokens_used": current_max_tokens,
                "attempt": attempt,
                "retry_reason": None,
            }

        except Exception as e:
            last_error = repr(e)
            last_retry_info = should_retry_after_parse_error(response, content)

            # If no retries remain, break to fallback.
            if attempt >= max_retries:
                break

            # Retry only because JSON was invalid/broken.
            continue

    # Last-resort fallback from raw content after all JSON retries failed.
    fallback = normalize_answer_text(strip_think_blocks(last_content or ""))

    # Avoid storing invalid verbose text as answer.
    if "{" in fallback or "}" in fallback or len(fallback.split()) > 30:
        fallback = "Information not available"

    return {
        "response": fallback,
        "raw_content": last_content,
        "usage": None,
        "parse_error": last_error,
        "retry_info": last_retry_info,
        "max_tokens_used": ANSWER_RETRY_MAX_TOKENS,
        "attempt": "fallback",
        "retry_reason": "invalid_json_after_retries",
    }

In [11]:
# cell 11
# Output I/O and resume helpers.

def atomic_write_json(path, data):
    # Atomic JSON write.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def load_existing_answer_map(path):
    # Load existing answers for resume.
    path = Path(path)

    if not path.exists():
        return {}

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

    if not isinstance(data, list):
        return {}

    out = {}

    for pos, rec in enumerate(data):
        if not isinstance(rec, dict):
            continue

        source_index = rec.get("source_index", pos)

        try:
            source_index = int(source_index)
        except Exception:
            source_index = pos

        out[source_index] = rec

    return out

def save_answer_map(path, answer_map):
    # Save answers sorted by source_index.
    ordered = [
        answer_map[idx]
        for idx in sorted(answer_map.keys())
    ]
    atomic_write_json(path, ordered)

def make_base_output_record(record):
    # Convert phase-2 record to phase-3 output schema.
    return {
        "source_index": int(record.get("source_index", 0)),
        "type": record.get("type"),
        "question": record.get("question"),
        "gt": record.get("answer"),
    }

def make_success_output_record(record, llm_result):
    # Build final answer record.
    out = make_base_output_record(record)
    out["response"] = llm_result["response"]
    out["attempt"] = llm_result.get("attempt")
    out["max_tokens_used"] = llm_result.get("max_tokens_used")
    out["completion_tokens"] = llm_result.get("completion_tokens")
    out["retry_reason"] = llm_result.get("retry_reason")
    return out

def make_error_output_record(record, error):
    # Build error record but keep response field.
    out = make_base_output_record(record)
    out["response"] = "Information not available"
    out["error"] = str(error)
    out["traceback"] = traceback.format_exc()
    return out

existing_answers = load_existing_answer_map(DRIVE_ANSWER_PATH)

print("Existing answer records:", len(existing_answers))
print("Answer output:", DRIVE_ANSWER_PATH)

Existing answer records: 481
Answer output: /content/drive/MyDrive/final_project/idea_1/answers/2wikimultihopqa/2wikimultihopqa_answer_qwen3.5.json


In [12]:
# cell 12
# Run phase 3 answer generation and save after each question.

start_idx = int(ANSWER_START_INDEX)
end_idx = len(evidence_records) if ANSWER_END_INDEX is None else int(ANSWER_END_INDEX)

run_records = evidence_records[start_idx:end_idx]

print("Total evidence records:", len(evidence_records))
print("Run range:", start_idx, "to", end_idx)
print("Existing answers:", len(existing_answers))
print("Output:", DRIVE_ANSWER_PATH)

progress = tqdm(run_records, desc="Generating answers", dynamic_ncols=True)

for local_pos, record in enumerate(progress, start=1):
    source_index = int(record.get("source_index", start_idx + local_pos - 1))

    # Skip completed records.
    if source_index in existing_answers:
        old = existing_answers[source_index]
        if isinstance(old, dict) and old.get("response") and "error" not in old:
            progress.set_postfix({
                "source_index": source_index,
                "status": "skipped",
                "saved": len(existing_answers),
            })
            continue

    try:
        prompt = build_answer_prompt(record)
        llm_result = call_llm_answer(prompt)

        output_record = make_success_output_record(record, llm_result)
        existing_answers[source_index] = output_record
        status = "ok"

    except Exception as e:
        output_record = make_error_output_record(record, e)
        existing_answers[source_index] = output_record
        status = "error"

    if local_pos % SAVE_EVERY_N == 0:
        save_answer_map(DRIVE_ANSWER_PATH, existing_answers)

    if local_pos % CLEAR_CACHE_EVERY_N == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    progress.set_postfix({
        "source_index": source_index,
        "status": status,
        "saved": len(existing_answers),
    })

save_answer_map(DRIVE_ANSWER_PATH, existing_answers)

print("Answer generation finished.")
print("Saved records:", len(existing_answers))
print("Output file:", DRIVE_ANSWER_PATH)

Total evidence records: 1000
Run range: 0 to 1000
Existing answers: 481
Output: /content/drive/MyDrive/final_project/idea_1/answers/2wikimultihopqa/2wikimultihopqa_answer_qwen3.5.json


Generating answers:   0%|          | 0/1000 [00:00<?, ?it/s]

Answer generation finished.
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/idea_1/answers/2wikimultihopqa/2wikimultihopqa_answer_qwen3.5.json


In [13]:
# cell 13
# Inspect saved answers.

with open(DRIVE_ANSWER_PATH, "r", encoding="utf-8") as f:
    saved_answers = json.load(f)

print("Saved answers:", len(saved_answers))
print("Output path:", DRIVE_ANSWER_PATH)

for rec in saved_answers[:5]:
    print("=" * 100)
    print("source_index:", rec.get("source_index"))
    print("type:", rec.get("type"))
    print("question:", rec.get("question"))
    print("gt:", rec.get("gt"))
    print("response:", rec.get("response"))

Saved answers: 1000
Output path: /content/drive/MyDrive/final_project/idea_1/answers/2wikimultihopqa/2wikimultihopqa_answer_qwen3.5.json
source_index: 0
type: comparison
question: Are North Marion High School (Oregon) and Seoul High School both located in the same country?
gt: no
response: No
source_index: 1
type: comparison
question: Are Fire In Hell and The Tiger: An Old Hunter'S Tale from the same country?
gt: yes
response: Yes, both are South Korean films.
source_index: 2
type: comparison
question: Are Simonds Catholic College and Saginaw High School (Texas) both located in the same country?
gt: no
response: No
source_index: 3
type: comparison
question: Who was born first, John Beach or Gordon Persons?
gt: John Beach
response: John Beach
source_index: 4
type: comparison
question: Who is older, Aryeh Ben-Eliezer or Jason Pociask?
gt: Aryeh Ben-Eliezer
response: Aryeh Ben-Eliezer
